# Imports y settings

In [ ]:
MODELS_IDS = [
    # Mistral
    "mistralai/Mistral-7B-v0.1",
    "mistralai/Mistral-7B-Instruct-v0.3",

    # OPT
    "facebook/opt-125m",
    "facebook/opt-1.3b",

    # Cohere
    "CohereForAI/aya-expanse-8b",

    # Llama 3.2
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B-Instruct",
    "meta-llama/Llama-3.2-3B",
    "meta-llama/Llama-3.2-3B-Instruct",

    # Llama 2
    "meta-llama/Llama-2-7b-hf",
    "meta-llama/Llama-2-7b-chat-hf",

    # Qwen3
    "Qwen/Qwen3-0.6B",
    "Qwen/Qwen3-1.7B",
    "Qwen/Qwen3-4B",
    "Qwen/Qwen3-8B",

    # Qwen2
    "Qwen/Qwen2-0.5B-Instruct",
    "Qwen/Qwen2-0.5B",
    "Qwen/Qwen2-1.5B",

    # GPT2
    "openai-community/gpt2-medium",
    "openai-community/gpt2",
    "openai-community/gpt2-xl",

    # SmolLM
    "HuggingFaceTB/SmolLM-135M",
    "HuggingFaceTB/SmolLM-135M-Instruct",
    "HuggingFaceTB/SmolLM-360M",
    "HuggingFaceTB/SmolLM-360M-Instruct",

    # SmolLM2
    "HuggingFaceTB/SmolLM2-135M",
    "HuggingFaceTB/SmolLM2-135M-Instruct",
    "HuggingFaceTB/SmolLM2-360M",
    "HuggingFaceTB/SmolLM2-360M-Instruct",

    # Gemma-3
    "google/gemma-3-4b-it",
    "google/gemma-3-4b-pt",
    "google/gemma-3-1b-it",
    "google/gemma-3-1b-pt",
    "google/gemma-3-270m-it",
    "google/gemma-3-270m",

    # Gemma-2
    "google/gemma-2-2b",
    "google/gemma-2-2b-it",

    # Gemma
    "google/gemma-2b",
    "google/gemma-2b-it",
]

DIR_PATH = "/content/drive/MyDrive/Tesis/"
DATASET_PATH = "input_scores.csv"
MODELS_PATH = "models"
RESULTS_DIR = "results-final"
CHECKPOINT_DIR = "checkpoints"
EXPERIMENT_NAME = "experimento"

prefixs = ("User location: Argentina\nUser language: es-AR", "User location: Spain\nUser language: es-ES")

In [ ]:
import pandas as pd
import os
import torch
import datetime
import logging
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
HF_HOME = os.path.join(DIR_PATH, MODELS_PATH)
os.environ["HF_HOME"] = HF_HOME
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
logging.basicConfig(
    level=logging.DEBUG,
    format="%(funcName)s - %(message)s",
    force=True
)

logger = logging.getLogger(__name__)
logger.setLevel("DEBUG")

log_timestamp = datetime.datetime.now().isoformat()
log_file_path = os.path.join(DIR_PATH, RESULTS_DIR, f"log_{log_timestamp}.log")

file_handler = logging.FileHandler(log_file_path)
file_handler.setLevel(logging.DEBUG) # Ensure debug messages are written to file

formatter = logging.Formatter("%(funcName)s - %(message)s")
file_handler.setFormatter(formatter)

# Add the handler to the logger
logger.addHandler(file_handler)

# Limpieza de datos

In [ ]:
df = pd.read_csv(os.path.join(DIR_PATH,DATASET_PATH), index_col = 0)
df['base'] = df['base'].replace(pd.NA, "")
df['pair'] = df['pair'].apply(eval)

control_df = pd.DataFrame(data = [{"base": "vivo en", "pair":("Madrid","Buenos Aires")}]) # Agrego par de control

df = pd.concat([control_df, df]).reset_index(drop = True)

df['pair'] = df['pair'].apply(lambda x: (x[1],x[0]))

# Funciones

In [ ]:
def tokens_to_tensor(token_ids):
    "Toma token_ids y los convierte en el formato que requiere transformers para hacer inferencia"
    tensor = {
        "input_ids": torch.tensor([token_ids]).to(device),
        "attention_mask": torch.ones(1, len(token_ids), dtype=torch.long).to(device)
    }
    return tensor

def get_logprob(token_ids, model, tokenizer):
    "Devuelve el log de la proba predicha para el último token de la oración"
    logger.debug(f"{tokenizer.convert_ids_to_tokens(token_ids)}")
    input = tokens_to_tensor(token_ids)
    with torch.no_grad():
        output = model(**input)
        logits = output.logits
    log_prob = torch.log_softmax(logits[0][-1], dim=-1)
    return log_prob

In [ ]:
def get_delta(gb: pd.DataFrame, model, tokenizer):
    "Función principal que se ocupa de obtener los scores"
    base = gb["base"].strip("\n ")
    pair = gb["pair"]
    assert isinstance(pair, tuple)
    accumulated_logprobs = {}
    tokens_count = {}

    for n_prefix, prefix in enumerate(prefixs):
        prefix = prefix.strip("\n ")
        try:
          prefix_tokens = tokenizer.apply_chat_template([{"role":"system", "content": prefix}, {"role":"user","content":base.strip()}], tokenize = True)['input_ids'][0:-2]
          if "system" not in tokenizer.decode(prefix_tokens):
            raise ValueError("Rol system no soportado")
        except Exception:
          logger.exception(f"Error generando chat template con {model}")
          prefix_tokens = tokenizer((prefix + "\nUser message: " + base).strip())['input_ids']

        accumulated_logprobs[n_prefix] = {}

        for n_pair, p in enumerate(pair):
            logger.debug(f'n_prefix: {n_prefix}, n_pair: {n_pair}')
            p = p.strip()
            full_sentence = tokenizer.decode(prefix_tokens) + " " + p
            accumulated_logprob = 0
            full_tokens = tokenizer.encode(full_sentence, add_special_tokens=False)
            logger.debug(f'Full sentence: {tokenizer.decode(full_tokens).replace("\n","\\n")}"')
            tokenized = full_tokens[len(prefix_tokens):]
            logger.debug(f"Tokenized: {tokenized}")
            tokens_count[n_pair] = len(tokenized)

            for i, _ in enumerate(tokenized):
                logprob = get_logprob(prefix_tokens + tokenized[0:i], model = model, tokenizer = tokenizer)[tokenized[i]]
                accumulated_logprob += logprob
                logger.debug(f"Logprob de {tokenizer.convert_ids_to_tokens(tokenized[i])}: {logprob}")

            accumulated_logprobs[n_prefix][n_pair] = accumulated_logprob

    acummulated_ar_ar = accumulated_logprobs[0][0].item()
    acummulated_ar_es = accumulated_logprobs[0][1].item()
    acummulated_es_ar = accumulated_logprobs[1][0].item()
    acummulated_es_es = accumulated_logprobs[1][1].item()
    diff_pre_ar = acummulated_ar_ar - acummulated_ar_es
    diff_pre_es =  acummulated_es_ar - acummulated_es_es
    delta = diff_pre_ar - diff_pre_es

    return pd.Series({"diff_pre_ar": diff_pre_ar,
                      "diff_pre_es":diff_pre_es,
                      "delta":delta,
                      "accumulated_ar": (acummulated_ar_ar, acummulated_ar_es),
                      "accumulated_es": (acummulated_es_ar, acummulated_es_es),
                      "token_count_ar": tokens_count[0],
                      "token_count_es": tokens_count[1],
                      "prefixes": (prefixs[0],prefixs[1]),
                      "full_sentence_debug": full_sentence.replace("\n","\\n")})

In [ ]:
def inference(model_id, resume_df=None):
  "Función orquestadora para levantar los LLMs, retomar del checkpoint (si existe) y ejecutar la función que obtiene los deltas"
  path = snapshot_download(
    repo_id=model_id,
    cache_dir=HF_HOME
  )
  tokenizer = AutoTokenizer.from_pretrained(path, local_files_only = True)
  model = AutoModelForCausalLM.from_pretrained(path, local_files_only = True, device_map="auto", offload_folder="offload_dir/")

  checkpoint_path = os.path.join(DIR_PATH, RESULTS_DIR, CHECKPOINT_DIR, f"{EXPERIMENT_NAME}_{model_id.replace('/', '_')}_checkpoint.csv")
  os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

  if resume_df is not None and not resume_df.empty:
    processed_indices = resume_df.index.tolist()
    rows_to_process = df[~df.index.isin(processed_indices)]
    current_results = resume_df.copy()
    print(f"Reanundando {model_id}: {len(processed_indices)} filas ya obtenidas. Quedan: {len(rows_to_process)}")
  else:
    rows_to_process = df
    current_results = pd.DataFrame()
    print(f"Empezando {model_id} from scratch: {len(rows_to_process)} filas para procesar.")

  for idx, row in rows_to_process.iterrows():
    logger.info(f"Procesando fila {idx} para {model_id}")

    delta_data = get_delta(row, tokenizer=tokenizer, model=model)

    row_result = pd.concat([pd.Series(row), delta_data]).to_frame().T
    row_result.index = [idx]
    row_result["model"] = model_id
    row_result["timestamp"] = datetime.datetime.now().isoformat()

    current_results = pd.concat([current_results, row_result])
    current_results.to_csv(checkpoint_path)

  return current_results

# Inference

In [ ]:
results_df = pd.DataFrame()

for model_id in MODELS_IDS:
  print(f"Empezando/Reanudando: {model_id}")

  final_model_path = os.path.join(DIR_PATH, RESULTS_DIR, f"final_{EXPERIMENT_NAME}_{model_id.replace('/', '_')}.csv")

  if os.path.exists(final_model_path):
    print(f"Ya existen resultados para {model_id}. Cargando desde {final_model_path}.")
    results = pd.read_csv(final_model_path, index_col=0)
  else:
    checkpoint_file = os.path.join(DIR_PATH, RESULTS_DIR, CHECKPOINT_DIR, f"{EXPERIMENT_NAME}_{model_id.replace('/', '_')}_checkpoint.csv")

    resume_data = None
    if EXPERIMENT_NAME and os.path.exists(checkpoint_file):
      resume_data = pd.read_csv(checkpoint_file, index_col=0)

    results = inference(model_id, resume_df=resume_data)

    results.to_csv(final_model_path)

  results_df = pd.concat([results_df, results])